# Task 8 — Spend-Quality Guardrail
**PlaceMux · Phase 2 · AI/ML Engineer**

**Goal**: Add an AI decision layer that warns students *before* they spend ₹100 on a job application where their match score is too low.

This notebook walks through:
1. Loading the data and building the matching model
2. Applying the spend-quality guardrail
3. Evaluating with real metrics (precision, recall, FPR)
4. Full end-to-end demo examples
5. Saving the experiment log

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from models.matching_model import MatchingModel, compute_match_score
from models.guardrail import apply_guardrail, THRESHOLDS

print("Imports OK")

Imports OK


## 1. Load Data

In [2]:
data_dir = Path("../data")
students_df = pd.read_csv(data_dir / "students.csv")
jobs_df = pd.read_csv(data_dir / "jobs.csv")

print(f"Students: {len(students_df)}")
print(f"Jobs    : {len(jobs_df)}")
students_df.head(3)

Students: 100
Jobs    : 40


,student_id,name,skills,experience_years,education
0,1,Student_001,"Excel, SQL, C++, R, Teamwork, NLP, REST API, D...",1.7,B.Tech
1,2,Student_002,"TensorFlow, Keras, Project Management, SQL",2.2,MBA
2,3,Student_003,"Teamwork, MongoDB, Keras, AWS, JavaScript, Sta...",1.4,BCA


## 2. Build & Save the Matching Model

In [3]:
model = MatchingModel(students_df, jobs_df)
model.save("../models/matching_model.pkl")
print("Model ready.")

Model saved to ../models/matching_model.pkl
Model ready.


## 3. Generate All Pairs for Evaluation

We compute match scores for every student × job combination so we have a real dataset to evaluate on.

In [4]:
records = []
for _, student in students_df.iterrows():
    for _, job in jobs_df.iterrows():
        result = compute_match_score(student["skills"], job["required_skills"])
        records.append({
            "student_id": student["student_id"],
            "job_id": job["job_id"],
            "student_name": student["name"],
            "job_title": job["title"],
            "match_score": result["match_score"],
            "n_matched": len(result["matched_skills"]),
            "n_missing": len(result["missing_skills"]),
            "matched_skills": ", ".join(result["matched_skills"]),
            "missing_skills": ", ".join(result["missing_skills"]),
        })

eval_df = pd.DataFrame(records)
print(f"Total pairs evaluated: {len(eval_df)}")
eval_df["match_score"].describe().round(1)

Total pairs evaluated: 4000


count    4000.0
mean       26.9
std        18.2
min         0.0
25%        14.3
50%        25.0
75%        40.0
max       100.0
Name: match_score, dtype: float64

## 4. Apply Guardrail Decision

In [5]:
def guardrail_status(score):
    for low, high, status, _, _ in THRESHOLDS:
        if low <= score < high:
            return status
    return "UNKNOWN"

eval_df["guardrail_status"] = eval_df["match_score"].apply(guardrail_status)
eval_df["allow_payment"] = eval_df["match_score"].apply(lambda s: s >= 50)

print("Guardrail decisions:")
print(eval_df["guardrail_status"].value_counts())
print(f"\nAllowed: {eval_df['allow_payment'].sum()}  |  Blocked: {(~eval_df['allow_payment']).sum()}")

Guardrail decisions:
guardrail_status
VERY_LOW_MATCH     1873
LOW_MATCH          1596
MODERATE_MATCH      472
GOOD_MATCH           48
EXCELLENT_MATCH      11
Name: count, dtype: int64

Allowed: 531  |  Blocked: 3469


## 5. Evaluation Metrics

We define a *true bad match* as score < 50 (our payment-block threshold).
- **Precision**: when we warn, are we right?
- **Recall**: do we catch all genuinely poor matches?
- **False Positive Rate**: how often do we wrongly warn a good match?

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = (eval_df["match_score"] < 50).astype(int)
y_pred = (~eval_df["allow_payment"]).astype(int)

precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print("=" * 52)
print("  GUARDRAIL METRICS")
print("=" * 52)
print(f"  Precision          : {precision:.3f}")
print(f"  Recall             : {recall:.3f}")
print(f"  F1 Score           : {f1:.3f}")
print(f"  False Positive Rate: {fpr:.3f}")
print()
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

# baseline: always warn everyone
import numpy as np
y_base = np.ones(len(y_true), dtype=int)
bp = precision_score(y_true, y_base)
br = recall_score(y_true, y_base)
bf = f1_score(y_true, y_base)
print(f"\n  Baseline (warn all) → P={bp:.3f}  R={br:.3f}  F1={bf:.3f}")
print(f"  Our model precision gain over baseline: +{(precision-bp)*100:.1f}pp")

  GUARDRAIL METRICS
  Precision          : 1.000
  Recall             : 1.000
  F1 Score           : 1.000
  False Positive Rate: 0.000

  TP=3469  FP=0  FN=0  TN=531

  Baseline (warn all) → P=0.867  R=1.000  F1=0.929
  Our model precision gain over baseline: +13.3pp


## 6. End-to-End Demo Examples

In [7]:
def demo(student_id, job_id):
    result = model.predict(student_id, job_id)
    decision = apply_guardrail(result)
    decision.display()

print("Example 1 — Poor match (payment should be blocked):")
demo(3, 2)

print("\nExample 2 — Strong match:")
best = eval_df.sort_values("match_score", ascending=False).iloc[0]
demo(int(best["student_id"]), int(best["job_id"]))

print("\nExample 3 — Borderline (just above threshold):")
borderline = eval_df[abs(eval_df["match_score"] - 52) < 5].iloc[0]
demo(int(borderline["student_id"]), int(borderline["job_id"]))

Example 1 — Poor match (payment should be blocked):

  Student_003  →  BI Developer @ Company_C
  Match Score  : 33.3%
  Status       : ⚠️  Low Match
  Decision     : 🚫 Payment blocked — see warning

  ✓ Matched    : power bi, statistics
  ✗ Missing    : data visualization, excel, sql, tableau

  💬 Low chance of shortlisting. We recommend building the missing skills first.
  📌 Advice    : You're missing data visualization, excel, sql. Consider upskilling before spending ₹100 on this application.

Example 2 — Strong match:

  Student_034  →  Data Engineer @ Company_F
  Match Score  : 100.0%
  Status       : ✅ Excellent Match
  Decision     : ✅ Payment allowed

  ✓ Matched    : aws, docker, pandas, postgresql, python, sql
  ✗ Missing    : None

  💬 Strong profile for this role. Apply confidently.
  📌 Advice    : You're well-positioned for this role. Go ahead and apply.

Example 3 — Borderline (just above threshold):

  Student_001  →  Data Scientist @ Company_P
  Match Score  : 50.0%
  S

## 7. Save Experiment Log

In [8]:
log_path = Path("../logs/experiment_log.csv")
log_path.parent.mkdir(exist_ok=True)
eval_df.to_csv(log_path, index=False)

metrics = {
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1_score": round(f1, 4),
    "false_positive_rate": round(fpr, 4),
    "true_positives": int(tp),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "total_pairs_evaluated": len(eval_df),
    "pairs_allowed": int(eval_df["allow_payment"].sum()),
    "pairs_blocked": int((~eval_df["allow_payment"]).sum()),
}
with open("../logs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Experiment log saved → {log_path}  ({len(eval_df)} rows)")
print("Metrics saved → logs/metrics.json")
print(json.dumps(metrics, indent=2))

Experiment log saved → ..\logs\experiment_log.csv  (4000 rows)
Metrics saved → logs/metrics.json
{
  "precision": 1.0,
  "recall": 1.0,
  "f1_score": 1.0,
  "false_positive_rate": 0.0,
  "true_positives": 3469,
  "true_negatives": 531,
  "false_positives": 0,
  "false_negatives": 0,
  "total_pairs_evaluated": 4000,
  "pairs_allowed": 531,
  "pairs_blocked": 3469
}
